In [ ]:
# import sgkit as sg
# import numpy as np

# # Load your zarr dataset
# ds = sg.load_dataset("path/to/your/data.zarr")

# # View available variables in your dataset
# print(ds.data_vars)

# ## Method 1: #####################################

# # Define your genomic region
# chrom = "22"  # or however chromosomes are encoded in your data
# start_pos = 16050000
# end_pos = 16100000

# # Filter by chromosome (if contig/chromosome info is available)
# if "variant_contig" in ds:
#     # Get chromosome index
#     chrom_idx = np.where(ds.contig_id.values == chrom)[0][0]
    
#     # Filter variants by chromosome and position
#     mask = (
#         (ds.variant_contig == chrom_idx) & 
#         (ds.variant_position >= start_pos) & 
#         (ds.variant_position <= end_pos)
#     )
# else:
#     # If no chromosome info, just filter by position
#     mask = (
#         (ds.variant_position >= start_pos) & 
#         (ds.variant_position <= end_pos)
#     )

# # Apply the filter
# region_ds = ds.sel(variants=mask)

# # Extract the information
# positions = region_ds.variant_position.values
# ref_alleles = region_ds.variant_allele.values[:, 0]  # First allele is reference
# alt_alleles = region_ds.variant_allele.values[:, 1:]  # Remaining are alternatives


# ## Method 1: #####################################

# # If your dataset has proper indexing, you can use query_variants
# # This requires variant_contig and variant_position to be properly set
# region_ds = sg.query_variants(
#     ds, 
#     region="22:16050000-16100000"  # Format: "chrom:start-end"
# )

# # Extract the data
# positions = region_ds.variant_position.values
# ref_alleles = region_ds.variant_allele.values[:, 0]
# alt_alleles = region_ds.variant_allele.values[:, 1:]


# ## Creating a Summary DataFrame

# import pandas as pd

# # Create a more readable output
# variant_info = []

# for i in range(len(positions)):
#     # Handle multiple alternative alleles
#     alts = alt_alleles[i]
#     # Filter out missing/padding alleles (often empty strings or specific missing values)
#     valid_alts = [alt for alt in alts if alt != '' and alt != '.']
    
#     variant_info.append({
#         'position': positions[i],
#         'ref': ref_alleles[i].decode() if isinstance(ref_alleles[i], bytes) else ref_alleles[i],
#         'alt': ','.join([alt.decode() if isinstance(alt, bytes) else alt for alt in valid_alts])
#     })

# df = pd.DataFrame(variant_info)
# print(df)

# ## Handling Common Data Formats


# # Some datasets store alleles differently
# if "variant_allele" in ds:
#     # Standard sgkit format
#     alleles = region_ds.variant_allele.values
# elif "allele" in ds:
#     # Alternative naming
#     alleles = region_ds.allele.values
    
# # Handle string encoding
# if alleles.dtype.kind == 'S' or alleles.dtype.kind == 'U':
#     # Already strings
#     ref = alleles[:, 0]
#     alt = alleles[:, 1:]
# else:
#     # Might need decoding
#     ref = np.char.decode(alleles[:, 0])
#     alt = np.char.decode(alleles[:, 1:])


###########################

def extract_variants_in_region(zarr_path, chrom, start, end):
    """
    Extract variant information from a genomic region.
    
    Parameters:
    -----------
    zarr_path : str
        Path to zarr dataset
    chrom : str
        Chromosome name
    start : int
        Start position
    end : int
        End position
    
    Returns:
    --------
    pandas.DataFrame with columns: chromosome, position, ref, alt
    """
    import sgkit as sg
    import pandas as pd
    import numpy as np
    
    # Load dataset
    ds = sg.load_dataset(zarr_path)
    
    # Filter to region
    if "variant_contig" in ds:
        # Map chromosome to index
        chrom_idx = np.where(ds.contig_id.values == chrom)[0][0]
        mask = (
            (ds.variant_contig == chrom_idx) & 
            (ds.variant_position >= start) & 
            (ds.variant_position <= end)
        )
    else:
        mask = (ds.variant_position >= start) & (ds.variant_position <= end)
    
    # Subset data
    region_ds = ds.sel(variants=mask)
    
    # Extract information
    results = []
    for i in range(len(region_ds.variants)):
        alleles = region_ds.variant_allele.values[i]
        ref = alleles[0]
        alts = [a for a in alleles[1:] if a != '' and a != '.']
        
        results.append({
            'chromosome': chrom,
            'position': int(region_ds.variant_position.values[i]),
            'ref': ref if isinstance(ref, str) else ref.decode(),
            'alt': ','.join([a if isinstance(a, str) else a.decode() for a in alts])
        })
    
    return pd.DataFrame(results)


import sgkit as sg
import numpy as np

# Usage
df = extract_variants_in_region(
    "path/to/data.zarr",
    chrom="22",
    start=16050000,
    end=16100000
)
print(df)



